# Ablation A4 — Không có External Features

**Mục đích**: Đánh giá đóng góp của nhóm `external` (`Promotion`, `Epidemic`) bằng cách loại bỏ hoàn toàn 2 features này khỏi input.

Dựa trên kết quả **Permutation Feature Importance**, nhóm `external` là nhóm quan trọng nhất ở horizon 7 (importance = +14.11%) và quan trọng thứ 2 ở horizon 14 (+9.80%) và 28 (+8.62%).

| Variant | External Features | Model |
|---------|-------------------|-------|
| **A4 (this)** | ❌ Không có `Promotion`, `Epidemic` | LSTM + Entity Embedding |
| Proposed | ✅ Có đầy đủ | LSTM + Entity Embedding |

**Horizons**: 7, 14, 28 ngày  
**Output**: `result/ablation_A4_no_external_summary.csv` + `result/ablation_A4_no_external_details.csv`

In [13]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
import warnings, os
import kagglehub

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

DATA_DIR  = kagglehub.dataset_download("atomicd/retail-store-inventory-and-demand-forecasting")
DATA_PATH = os.path.join(DATA_DIR, "sales_data.csv")
RESULT_DIR = "/kaggle/working/"

TARGET    = 'Units Sold'
TRAIN_END = '2023-06-30'
VAL_END   = '2023-10-31'
HORIZONS  = [7, 14, 28]
LOOKBACK  = 30
LAG       = 7

# ── Ablation tag ─────────────────────────────────────────────────────────────
ABLATION_NAME = 'A4-NoExternal'
print(f'Ablation: {ABLATION_NAME}')

Ablation: A4-NoExternal


## 1. Load & Feature Engineering (không có External)

In [14]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# ── Static categoricals → encode cho Entity Embedding ────────────────────────
CAT_COLS = ['Store ID', 'Product ID', 'Category', 'Region']
cat_vocabs = {}
for col in CAT_COLS:
    uniq = sorted(df_raw[col].unique())
    cat_vocabs[col] = {v: i for i, v in enumerate(uniq)}
    df_raw[col + '_enc'] = df_raw[col].map(cat_vocabs[col])

# ── Time-varying categoricals → ordinal encode ───────────────────────────────
WEATHER_MAP = {'Sunny': 0, 'Cloudy': 1, 'Rainy': 2, 'Snowy': 3, 'Windy': 4, 'Stormy': 5}
SEASON_MAP  = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Fall': 3}
df_raw['weather_enc'] = df_raw['Weather Condition'].map(WEATHER_MAP).fillna(0).astype(int)
df_raw['season_enc']  = df_raw['Seasonality'].map(SEASON_MAP).fillna(0).astype(int)

# ── Lag & rolling features ────────────────────────────────────────────────────
grp = df_raw.groupby(['Store ID', 'Product ID'])[TARGET]
df_raw['lag_7']           = grp.shift(7)
df_raw['lag_14']          = grp.shift(14)
df_raw['lag_28']          = grp.shift(28)
df_raw['rolling_mean_7']  = grp.transform(lambda x: x.shift(1).rolling(7).mean())
df_raw['rolling_mean_14'] = grp.transform(lambda x: x.shift(1).rolling(14).mean())
df_raw['day_of_week']     = df_raw['Date'].dt.dayofweek
df_raw['day_of_month']    = df_raw['Date'].dt.day
df_raw['month']           = df_raw['Date'].dt.month
df_raw['is_weekend']      = (df_raw['day_of_week'] >= 5).astype(int)
df_raw = df_raw.bfill().fillna(0)

# ── NUM_COLS: bỏ 'Promotion' và 'Epidemic' ───────────────────────────────────
NUM_COLS = [
    TARGET,
    'Price', 'Discount', 'Competitor Pricing',
    'Inventory Level', 'Units Ordered',
    # 'Promotion', 'Epidemic',   ← ABLATED
    'weather_enc', 'season_enc',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14',
    'day_of_week', 'day_of_month', 'month', 'is_weekend',
]
TARGET_IDX      = NUM_COLS.index(TARGET)
ENC_COLS        = [c + '_enc' for c in CAT_COLS]
cat_vocab_sizes = {col: len(cat_vocabs[col]) for col in CAT_COLS}
series_keys     = sorted(df_raw.groupby(['Store ID', 'Product ID']).groups.keys())

print(f'Series: {len(series_keys)} | Num features: {len(NUM_COLS)} (Promotion & Epidemic ablated)')
print(f'NUM_COLS: {NUM_COLS}')

Series: 100 | Num features: 17 (Promotion & Epidemic ablated)
NUM_COLS: ['Units Sold', 'Price', 'Discount', 'Competitor Pricing', 'Inventory Level', 'Units Ordered', 'weather_enc', 'season_enc', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'day_of_week', 'day_of_month', 'month', 'is_weekend']


## 2. Model — LSTM + Entity Embedding (giống Proposed, chỉ khác NUM_COLS)

In [15]:
def build_model(lookback, n_num, cat_vocab_sizes, horizon, lstm_units=64, dropout=0.2):
    """
    Giống proposed: LSTM + Entity Embedding.
    Chỉ khác: n_num nhỏ hơn vì đã bỏ Promotion & Epidemic.
    """
    num_input = layers.Input(shape=(lookback, n_num), name='num_input')

    cat_inputs, cat_embeds = [], []
    for i, (col, vocab_size) in enumerate(cat_vocab_sizes.items()):
        inp = layers.Input(shape=(1,), name=f'cat_{i}', dtype='int32')
        emb = layers.Embedding(vocab_size, min(50, (vocab_size + 1) // 2),
                               name='emb_' + col.replace(' ', '_'))(inp)
        emb = layers.Flatten()(emb)
        cat_inputs.append(inp)
        cat_embeds.append(emb)

    cat_concat = layers.Concatenate()(cat_embeds) if len(cat_embeds) > 1 else cat_embeds[0]
    cat_tiled  = layers.RepeatVector(lookback)(cat_concat)

    x = layers.Concatenate(axis=-1)([num_input, cat_tiled])
    x = layers.LSTM(lstm_units, return_sequences=True)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(lstm_units // 2)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(horizon)(x)

    model = models.Model(inputs=[num_input] + cat_inputs, outputs=out)
    model.compile(optimizer='adam', loss='mse')
    return model

demo = build_model(LOOKBACK, len(NUM_COLS), cat_vocab_sizes, horizon=7)
demo.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cat_0 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_1 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_2 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_3 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Store_ID        │ (None, 1, 3)      │         15 │ cat_0[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Product_ID      │ (None, 1, 10)     │        200 │ cat_1[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Category        │ (None, 1, 3)      │         15 │ cat_2[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Region          │ (None, 1, 2)      │          8 │ cat_3[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_20          │ (None, 3)         │          0 │ emb_Store_ID[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_21          │ (None, 10)        │          0 │ emb_Product_ID[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_22          │ (None, 3)         │          0 │ emb_Category[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_23          │ (None, 2)         │          0 │ emb_Region[0][0]  │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_10      │ (None, 18)        │          0 │ flatten_20[0][0], │
│ (Concatenate)       │                   │            │ flatten_21[0][0], │
│                     │                   │            │ flatten_22[0][0], │
│                     │                   │            │ flatten_23[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_input           │ (None, 30, 17)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_5     │ (None, 30, 18)    │          0 │ concatenate_10[0… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_11      │ (None, 30, 35)    │          0 │ num_input[0][0],  │
│ (Concatenate)       │                   │            │ repeat_vector_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, 30, 64)    │     25,600 │ concatenate_11[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 30, 64)    │          0 │ lstm_10[0][0]   

 Total params: 38,485 (150.33 KB)

 Trainable params: 38,485 (150.33 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Build Dataset

In [16]:
def make_sequences(num_arr, cat_row, lookback, horizon, stride=7):
    X_num, y = [], []
    for i in range(lookback, len(num_arr) - horizon + 1, stride):
        X_num.append(num_arr[i - lookback:i])
        y.append(num_arr[i:i + horizon, TARGET_IDX])
    X_num = np.array(X_num, dtype=np.float32)
    y     = np.array(y, dtype=np.float32)
    n     = len(X_num)
    X_cats = [np.full(n, cat_row[j], dtype=np.int32) for j in range(len(cat_row))]
    return X_num, X_cats, y


def build_global_arrays(horizon, lookback):
    X_num_tr, X_num_vl = [], []
    y_tr, y_vl = [], []
    X_cats_tr = [[] for _ in CAT_COLS]
    X_cats_vl = [[] for _ in CAT_COLS]
    scalers = {}

    for store, product in series_keys:
        sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)]
        sdf = sdf.set_index('Date')
        key = f'{store}_{product}'

        cat_row   = sdf[ENC_COLS].iloc[0].values.astype(np.int32)
        train_num = sdf[:TRAIN_END][NUM_COLS].values.astype(np.float32)
        val_num   = sdf[:VAL_END][NUM_COLS].values.astype(np.float32)

        scaler = StandardScaler().fit(train_num)
        scalers[key] = scaler

        Xn_tr, Xc_tr, yt = make_sequences(scaler.transform(train_num), cat_row, lookback, horizon)
        Xn_vl, Xc_vl, yv = make_sequences(scaler.transform(val_num),   cat_row, lookback, horizon)

        X_num_tr.append(Xn_tr); X_num_vl.append(Xn_vl)
        y_tr.append(yt);        y_vl.append(yv)
        for j in range(len(CAT_COLS)):
            X_cats_tr[j].append(Xc_tr[j])
            X_cats_vl[j].append(Xc_vl[j])

    return (
        np.concatenate(X_num_tr), [np.concatenate(x) for x in X_cats_tr], np.concatenate(y_tr),
        np.concatenate(X_num_vl), [np.concatenate(x) for x in X_cats_vl], np.concatenate(y_vl),
        scalers
    )

print('Dataset builder ready.')

Dataset builder ready.


## 4. Rolling Evaluation

In [17]:
def rolling_eval(model, scaler, store, product, horizon, lookback):
    sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)].set_index('Date')

    cat_row     = sdf[ENC_COLS].iloc[0].values.astype(np.int32)
    full_scaled = scaler.transform(sdf[NUM_COLS].values.astype(np.float32))
    eval_start  = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)
    eval_end    = sdf.index.max()

    all_fc, all_ac = [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        t_loc     = sdf.index.get_loc(t)
        win_start = t_loc - lookback
        if win_start < 0:
            t += pd.Timedelta(days=horizon); continue
        actual = sdf[TARGET][t: t + pd.Timedelta(days=horizon - 1)].values
        if len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue

        X_num  = full_scaled[win_start:t_loc][np.newaxis]          # (1, lookback, n_num)
        X_cats = [np.array([[cat_row[j]]], dtype=np.int32) for j in range(len(CAT_COLS))]

        fc_scaled = model.predict([X_num] + X_cats, verbose=0)[0]  # (horizon,)
        dummy = np.zeros((horizon, len(NUM_COLS)), dtype=np.float32)
        dummy[:, TARGET_IDX] = fc_scaled
        fc = np.clip(scaler.inverse_transform(dummy)[:, TARGET_IDX], 0, None)

        all_fc.append(fc)
        all_ac.append(actual.astype(np.float32))
        t += pd.Timedelta(days=horizon)

    if not all_fc:
        return {k: np.nan for k in ['smape', 'mase', 'rmse', 'rmsle']}

    fc_arr, ac_arr = np.array(all_fc), np.array(all_ac)
    train_vals = sdf[TARGET][:TRAIN_END].values.astype(np.float32)
    lag   = min(LAG, len(train_vals) - 1)
    denom = np.mean(np.abs(train_vals[lag:] - train_vals[:-lag])) or 1.0

    return {
        'smape': (2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100,
        'mase' : np.mean(np.abs(fc_arr - ac_arr)) / denom,
        'rmse' : float(np.sqrt(np.mean((fc_arr - ac_arr) ** 2))),
        'rmsle': float(np.sqrt(np.mean((np.log1p(np.clip(fc_arr, 0, None)) - np.log1p(np.clip(ac_arr, 0, None))) ** 2))),
    }

print('Evaluation function ready.')

Evaluation function ready.


## 5. Train & Evaluate

In [18]:
os.makedirs(RESULT_DIR, exist_ok=True)

summary_rows = []
detail_rows  = []

for h in HORIZONS:
    print(f'\n=== Horizon = {h} ===')
    print('  Building arrays...')
    X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers = build_global_arrays(h, LOOKBACK)
    print(f'  Train: {X_num_tr.shape} | Val: {X_num_vl.shape}')

    model = build_model(LOOKBACK, len(NUM_COLS), cat_vocab_sizes, horizon=h)

    cb = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(
        [X_num_tr] + X_cats_tr, y_tr,
        validation_data=([X_num_vl] + X_cats_vl, y_vl),
        epochs=50, batch_size=256,
        callbacks=[cb], verbose=0
    )

    print('  Rolling eval on TEST...')
    scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}
    for store, product in series_keys:
        key = f'{store}_{product}'
        r   = rolling_eval(model, scalers[key], store, product, h, LOOKBACK)
        for k in scores: scores[k].append(r[k])
        print(f"    {store} | {product} | sMAPE={r['smape']:.2f}% MASE={r['mase']:.4f} RMSE={r['rmse']:.2f} RMSLE={r['rmsle']:.4f}")
        detail_rows.append({
            'ablation': ABLATION_NAME,
            'model': 'LSTM-NoExternal',
            'store': store, 'product': product,
            'horizon': h, 'lookback': LOOKBACK,
            'smape': round(float(r['smape']), 4),
            'mase':  round(float(r['mase']),  4),
            'rmse':  round(float(r['rmse']),  4),
            'rmsle': round(float(r['rmsle']), 4),
        })

    row = {
        'ablation':      ABLATION_NAME,
        'model':         'LSTM-NoExternal',
        'dataset':       'retail_inventory_daily',
        'target':        TARGET,
        'horizon':       h,
        'lookback':      LOOKBACK,
        'mean_smape':    round(float(np.nanmean(scores['smape'])), 4),
        'median_smape':  round(float(np.nanmedian(scores['smape'])), 4),
        'mean_mase':     round(float(np.nanmean(scores['mase'])), 4),
        'median_mase':   round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmse':     round(float(np.nanmean(scores['rmse'])), 4),
        'median_rmse':   round(float(np.nanmedian(scores['rmse'])), 4),
        'mean_rmsle':    round(float(np.nanmean(scores['rmsle'])), 4),
        'median_rmsle':  round(float(np.nanmedian(scores['rmsle'])), 4),
    }
    summary_rows.append(row)
    print(f"  H={h} | sMAPE={row['mean_smape']:.2f}% MASE={row['mean_mase']:.4f} RMSE={row['mean_rmse']:.2f} RMSLE={row['mean_rmsle']:.4f}")

# ── Save ─────────────────────────────────────────────────────────────────────
summary_path = f'{RESULT_DIR}/ablation_A4_no_external_summary.csv'
detail_path  = f'{RESULT_DIR}/ablation_A4_no_external_details.csv'
pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
pd.DataFrame(detail_rows).to_csv(detail_path,   index=False)
print(f'\nSaved summary → {summary_path}')
print(f'Saved details → {detail_path}')
pd.DataFrame(summary_rows)


=== Horizon = 7 ===
  Building arrays...
  Train: (7300, 30, 17) | Val: (9100, 30, 17)
  Rolling eval on TEST...
    S001 | P0001 | sMAPE=36.83% MASE=0.8952 RMSE=40.17 RMSLE=0.4891
    S001 | P0002 | sMAPE=30.71% MASE=0.6864 RMSE=35.12 RMSLE=0.4370
    S001 | P0003 | sMAPE=35.88% MASE=0.7425 RMSE=49.88 RMSLE=0.5444
    S001 | P0004 | sMAPE=39.84% MASE=0.7915 RMSE=36.34 RMSLE=0.7116
    S001 | P0005 | sMAPE=38.00% MASE=0.7607 RMSE=45.46 RMSLE=0.5012
    S001 | P0006 | sMAPE=52.80% MASE=0.8911 RMSE=49.36 RMSLE=0.9654
    S001 | P0007 | sMAPE=44.84% MASE=0.8805 RMSE=55.45 RMSLE=0.6734
    S001 | P0008 | sMAPE=34.67% MASE=0.7768 RMSE=31.89 RMSLE=0.4667
    S001 | P0009 | sMAPE=33.35% MASE=0.8917 RMSE=45.10 RMSLE=0.4232
    S001 | P0010 | sMAPE=37.89% MASE=0.8212 RMSE=32.70 RMSLE=0.5111
    S001 | P0011 | sMAPE=40.50% MASE=0.8330 RMSE=32.68 RMSLE=0.5499
    S001 | P0012 | sMAPE=39.37% MASE=0.7869 RMSE=46.95 RMSLE=0.5791
    S001 | P0013 | sMAPE=46.86% MASE=0.9416 RMSE=41.73 RMSLE=0.7014
  

,ablation,model,dataset,target,horizon,lookback,mean_smape,median_smape,mean_mase,median_mase,mean_rmse,median_rmse,mean_rmsle,median_rmsle
0,A4-NoExternal,LSTM-NoExternal,retail_inventory_daily,Units Sold,7,30,39.3580,39.9414,0.8059,0.8070,41.3061,41.7421,0.6283,0.6209
1,A4-NoExternal,LSTM-NoExternal,retail_inventory_daily,Units Sold,14,30,38.9069,39.1505,0.7849,0.7849,40.4100,41.0442,0.6247,0.5893
2,A4-NoExternal,LSTM-NoExternal,retail_inventory_daily,Units Sold,28,30,38.6915,38.8930,0.7797,0.7824,40.0941,40.1514,0.6229,0.5981


## 6. So sánh với Proposed (Entity Embedding đầy đủ — GAO)

In [19]:
import glob

# Load proposed result (best tuned — GAO)
proposed_files = glob.glob(f'{RESULT_DIR}/*gao*summary*') + glob.glob(f'{RESULT_DIR}/*entity_emb_gao*')
if proposed_files:
    df_proposed = pd.read_csv(proposed_files[0])
    df_proposed['ablation'] = 'Proposed (EntityEmb-GAO)'
else:
    print('Proposed result not found — skipping comparison.')
    df_proposed = None

df_a4 = pd.DataFrame(summary_rows)

compare_cols = ['ablation', 'horizon', 'mean_smape', 'mean_mase', 'mean_rmse', 'mean_rmsle']

if df_proposed is not None:
    df_compare = pd.concat([
        df_a4[compare_cols],
        df_proposed[compare_cols]
    ], ignore_index=True).sort_values(['horizon', 'ablation'])
    print('\n=== Ablation Comparison: A4 (No External) vs Proposed ===')
    print(df_compare.to_string(index=False))
    print('\n→ sMAPE tăng khi bỏ External = đóng góp thực sự của Promotion & Epidemic')
else:
    print(df_a4[compare_cols].to_string(index=False))

Proposed result not found — skipping comparison.
     ablation  horizon  mean_smape  mean_mase  mean_rmse  mean_rmsle
A4-NoExternal        7     39.3580     0.8059    41.3061      0.6283
A4-NoExternal       14     38.9069     0.7849    40.4100      0.6247
A4-NoExternal       28     38.6915     0.7797    40.0941      0.6229


---
# Ablation A5 — Không có Calendar Features

**Mục đích**: Đánh giá đóng góp của nhóm `calendar` (`day_of_week`, `day_of_month`, `month`, `is_weekend`) bằng cách loại bỏ hoàn toàn 4 features này khỏi input.

Dựa trên kết quả **Permutation Feature Importance**, nhóm `calendar` là nhóm quan trọng nhất ở horizon 14 (+12.46%) và 28 (+10.25%), và quan trọng thứ 2 ở horizon 7 (+13.98%).

| Variant | Calendar Features | Model |
|---------|-------------------|-------|
| **A5 (this)** | ❌ Không có `day_of_week`, `day_of_month`, `month`, `is_weekend` | LSTM + Entity Embedding |
| Proposed | ✅ Có đầy đủ | LSTM + Entity Embedding |

**Output**: `result/ablation_A5_no_calendar_summary.csv` + `result/ablation_A5_no_calendar_details.csv`

In [20]:
ABLATION_NAME_A5 = 'A5-NoCalendar'
print(f'Ablation: {ABLATION_NAME_A5}')

Ablation: A5-NoCalendar


## A5.1 — Feature Engineering (không có Calendar)

In [21]:
# ── NUM_COLS_A5: bỏ calendar features ────────────────────────────────────────
NUM_COLS_A5 = [
    TARGET,
    'Price', 'Discount', 'Competitor Pricing',
    'Inventory Level', 'Units Ordered',
    'Promotion', 'Epidemic',
    'weather_enc', 'season_enc',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14',
    # 'day_of_week', 'day_of_month', 'month', 'is_weekend',  ← ABLATED
]
TARGET_IDX_A5 = NUM_COLS_A5.index(TARGET)

print(f'Num features A5: {len(NUM_COLS_A5)} (calendar ablated)')
print(f'NUM_COLS_A5: {NUM_COLS_A5}')

Num features A5: 15 (calendar ablated)
NUM_COLS_A5: ['Units Sold', 'Price', 'Discount', 'Competitor Pricing', 'Inventory Level', 'Units Ordered', 'Promotion', 'Epidemic', 'weather_enc', 'season_enc', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14']


In [22]:
def make_sequences_a5(num_arr, cat_row, lookback, horizon, stride=7):
    X_num, y = [], []
    for i in range(lookback, len(num_arr) - horizon + 1, stride):
        X_num.append(num_arr[i - lookback:i])
        y.append(num_arr[i:i + horizon, TARGET_IDX_A5])
    X_num = np.array(X_num, dtype=np.float32)
    y     = np.array(y, dtype=np.float32)
    n     = len(X_num)
    X_cats = [np.full(n, cat_row[j], dtype=np.int32) for j in range(len(cat_row))]
    return X_num, X_cats, y


def build_global_arrays_a5(horizon, lookback):
    X_num_tr, X_num_vl = [], []
    y_tr, y_vl = [], []
    X_cats_tr = [[] for _ in CAT_COLS]
    X_cats_vl = [[] for _ in CAT_COLS]
    scalers = {}

    for store, product in series_keys:
        sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)]
        sdf = sdf.set_index('Date')
        key = f'{store}_{product}'

        cat_row   = sdf[ENC_COLS].iloc[0].values.astype(np.int32)
        train_num = sdf[:TRAIN_END][NUM_COLS_A5].values.astype(np.float32)
        val_num   = sdf[:VAL_END][NUM_COLS_A5].values.astype(np.float32)

        scaler = StandardScaler().fit(train_num)
        scalers[key] = scaler

        Xn_tr, Xc_tr, yt = make_sequences_a5(scaler.transform(train_num), cat_row, lookback, horizon)
        Xn_vl, Xc_vl, yv = make_sequences_a5(scaler.transform(val_num),   cat_row, lookback, horizon)

        X_num_tr.append(Xn_tr); X_num_vl.append(Xn_vl)
        y_tr.append(yt);        y_vl.append(yv)
        for j in range(len(CAT_COLS)):
            X_cats_tr[j].append(Xc_tr[j])
            X_cats_vl[j].append(Xc_vl[j])

    return (
        np.concatenate(X_num_tr), [np.concatenate(x) for x in X_cats_tr], np.concatenate(y_tr),
        np.concatenate(X_num_vl), [np.concatenate(x) for x in X_cats_vl], np.concatenate(y_vl),
        scalers
    )

print('A5 dataset builder ready.')

A5 dataset builder ready.


In [23]:
def rolling_eval_a5(model, scaler, store, product, horizon, lookback):
    sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)].set_index('Date')

    cat_row     = sdf[ENC_COLS].iloc[0].values.astype(np.int32)
    full_scaled = scaler.transform(sdf[NUM_COLS_A5].values.astype(np.float32))
    eval_start  = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)
    eval_end    = sdf.index.max()

    all_fc, all_ac = [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        t_loc     = sdf.index.get_loc(t)
        win_start = t_loc - lookback
        if win_start < 0:
            t += pd.Timedelta(days=horizon); continue
        actual = sdf[TARGET][t: t + pd.Timedelta(days=horizon - 1)].values
        if len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue

        X_num  = full_scaled[win_start:t_loc][np.newaxis]
        X_cats = [np.array([[cat_row[j]]], dtype=np.int32) for j in range(len(CAT_COLS))]

        fc_scaled = model.predict([X_num] + X_cats, verbose=0)[0]
        dummy = np.zeros((horizon, len(NUM_COLS_A5)), dtype=np.float32)
        dummy[:, TARGET_IDX_A5] = fc_scaled
        fc = np.clip(scaler.inverse_transform(dummy)[:, TARGET_IDX_A5], 0, None)

        all_fc.append(fc)
        all_ac.append(actual.astype(np.float32))
        t += pd.Timedelta(days=horizon)

    if not all_fc:
        return {k: np.nan for k in ['smape', 'mase', 'rmse', 'rmsle']}

    fc_arr, ac_arr = np.array(all_fc), np.array(all_ac)
    train_vals = sdf[TARGET][:TRAIN_END].values.astype(np.float32)
    lag   = min(LAG, len(train_vals) - 1)
    denom = np.mean(np.abs(train_vals[lag:] - train_vals[:-lag])) or 1.0

    return {
        'smape': (2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100,
        'mase' : np.mean(np.abs(fc_arr - ac_arr)) / denom,
        'rmse' : float(np.sqrt(np.mean((fc_arr - ac_arr) ** 2))),
        'rmsle': float(np.sqrt(np.mean((np.log1p(np.clip(fc_arr, 0, None)) - np.log1p(np.clip(ac_arr, 0, None))) ** 2))),
    }

print('A5 evaluation function ready.')

A5 evaluation function ready.


In [24]:
summary_rows_a5 = []
detail_rows_a5  = []

for h in HORIZONS:
    print(f'\n=== A5 | Horizon = {h} ===')
    X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers_a5 = build_global_arrays_a5(h, LOOKBACK)
    print(f'  Train: {X_num_tr.shape} | Val: {X_num_vl.shape}')

    model_a5 = build_model(LOOKBACK, len(NUM_COLS_A5), cat_vocab_sizes, horizon=h)

    cb = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model_a5.fit(
        [X_num_tr] + X_cats_tr, y_tr,
        validation_data=([X_num_vl] + X_cats_vl, y_vl),
        epochs=50, batch_size=256,
        callbacks=[cb], verbose=0
    )

    print('  Rolling eval on TEST...')
    scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}
    for store, product in series_keys:
        key = f'{store}_{product}'
        r   = rolling_eval_a5(model_a5, scalers_a5[key], store, product, h, LOOKBACK)
        for k in scores: scores[k].append(r[k])
        print(f"    {store} | {product} | sMAPE={r['smape']:.2f}% MASE={r['mase']:.4f} RMSE={r['rmse']:.2f} RMSLE={r['rmsle']:.4f}")
        detail_rows_a5.append({
            'ablation': ABLATION_NAME_A5,
            'model': 'LSTM-NoCalendar',
            'store': store, 'product': product,
            'horizon': h, 'lookback': LOOKBACK,
            'smape': round(float(r['smape']), 4),
            'mase':  round(float(r['mase']),  4),
            'rmse':  round(float(r['rmse']),  4),
            'rmsle': round(float(r['rmsle']), 4),
        })

    row = {
        'ablation':      ABLATION_NAME_A5,
        'model':         'LSTM-NoCalendar',
        'dataset':       'retail_inventory_daily',
        'target':        TARGET,
        'horizon':       h,
        'lookback':      LOOKBACK,
        'mean_smape':    round(float(np.nanmean(scores['smape'])), 4),
        'median_smape':  round(float(np.nanmedian(scores['smape'])), 4),
        'mean_mase':     round(float(np.nanmean(scores['mase'])), 4),
        'median_mase':   round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmse':     round(float(np.nanmean(scores['rmse'])), 4),
        'median_rmse':   round(float(np.nanmedian(scores['rmse'])), 4),
        'mean_rmsle':    round(float(np.nanmean(scores['rmsle'])), 4),
        'median_rmsle':  round(float(np.nanmedian(scores['rmsle'])), 4),
    }
    summary_rows_a5.append(row)
    print(f"  H={h} | sMAPE={row['mean_smape']:.2f}% MASE={row['mean_mase']:.4f} RMSE={row['mean_rmse']:.2f} RMSLE={row['mean_rmsle']:.4f}")

# ── Save ─────────────────────────────────────────────────────────────────────
summary_path_a5 = f'{RESULT_DIR}/ablation_A5_no_calendar_summary.csv'
detail_path_a5  = f'{RESULT_DIR}/ablation_A5_no_calendar_details.csv'
pd.DataFrame(summary_rows_a5).to_csv(summary_path_a5, index=False)
pd.DataFrame(detail_rows_a5).to_csv(detail_path_a5,   index=False)
print(f'\nSaved summary → {summary_path_a5}')
print(f'Saved details → {detail_path_a5}')
pd.DataFrame(summary_rows_a5)


=== A5 | Horizon = 7 ===
  Train: (7300, 30, 15) | Val: (9100, 30, 15)
  Rolling eval on TEST...
    S001 | P0001 | sMAPE=35.79% MASE=0.8708 RMSE=39.60 RMSLE=0.4674
    S001 | P0002 | sMAPE=31.50% MASE=0.7048 RMSE=35.96 RMSLE=0.4408
    S001 | P0003 | sMAPE=39.91% MASE=0.8485 RMSE=57.05 RMSLE=0.6081
    S001 | P0004 | sMAPE=40.70% MASE=0.7940 RMSE=37.52 RMSLE=0.6915
    S001 | P0005 | sMAPE=35.84% MASE=0.7124 RMSE=43.56 RMSLE=0.4655
    S001 | P0006 | sMAPE=50.60% MASE=0.8290 RMSE=47.73 RMSLE=0.9093
    S001 | P0007 | sMAPE=40.93% MASE=0.7778 RMSE=53.62 RMSLE=0.6029
    S001 | P0008 | sMAPE=38.49% MASE=0.8437 RMSE=35.47 RMSLE=0.5002
    S001 | P0009 | sMAPE=31.81% MASE=0.8581 RMSE=45.41 RMSLE=0.4200
    S001 | P0010 | sMAPE=38.85% MASE=0.7923 RMSE=31.54 RMSLE=0.4758
    S001 | P0011 | sMAPE=38.59% MASE=0.7947 RMSE=34.65 RMSLE=0.5301
    S001 | P0012 | sMAPE=38.77% MASE=0.7436 RMSE=44.41 RMSLE=0.5241
    S001 | P0013 | sMAPE=45.75% MASE=0.8977 RMSE=39.98 RMSLE=0.6386
    S001 | P0014 |

,ablation,model,dataset,target,horizon,lookback,mean_smape,median_smape,mean_mase,median_mase,mean_rmse,median_rmse,mean_rmsle,median_rmsle
0,A5-NoCalendar,LSTM-NoCalendar,retail_inventory_daily,Units Sold,7,30,39.5124,39.6568,0.7997,0.8024,41.6875,41.2379,0.6113,0.6076
1,A5-NoCalendar,LSTM-NoCalendar,retail_inventory_daily,Units Sold,14,30,41.0202,40.4710,0.8127,0.8088,42.2210,42.1948,0.6279,0.6002
2,A5-NoCalendar,LSTM-NoCalendar,retail_inventory_daily,Units Sold,28,30,39.0638,38.3001,0.7823,0.7684,40.5398,40.7954,0.6161,0.5933


In [25]:
compare_cols = ['ablation', 'horizon', 'mean_smape', 'mean_mase', 'mean_rmse', 'mean_rmsle']

frames = [df_a4[compare_cols], pd.DataFrame(summary_rows_a5)[compare_cols]]
if df_proposed is not None:
    frames.append(df_proposed[compare_cols])

df_all_compare = pd.concat(frames, ignore_index=True).sort_values(['horizon', 'ablation'])
print('\n=== Ablation Comparison: A4 / A5 / Proposed ===')
print(df_all_compare.to_string(index=False))
print('\n→ sMAPE tăng khi bỏ Calendar = đóng góp thực sự của day_of_week / month / is_weekend')


=== Ablation Comparison: A4 / A5 / Proposed ===
     ablation  horizon  mean_smape  mean_mase  mean_rmse  mean_rmsle
A4-NoExternal        7     39.3580     0.8059    41.3061      0.6283
A5-NoCalendar        7     39.5124     0.7997    41.6875      0.6113
A4-NoExternal       14     38.9069     0.7849    40.4100      0.6247
A5-NoCalendar       14     41.0202     0.8127    42.2210      0.6279
A4-NoExternal       28     38.6915     0.7797    40.0941      0.6229
A5-NoCalendar       28     39.0638     0.7823    40.5398      0.6161

→ sMAPE tăng khi bỏ Calendar = đóng góp thực sự của day_of_week / month / is_weekend
